In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
PROJECT_ROOT = "/content/drive/MyDrive/FabricDefect"
print(PROJECT_ROOT)


/content/drive/MyDrive/FabricDefect


In [ ]:
!pip install kaggle


In [ ]:
!mkdir -p ~/.kaggle
from google.colab import files
files.upload()   # upload kaggle.json manually
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json



Saving kaggle.json to kaggle.json


**Download Dataset From Kaggle**

In [ ]:
!ls -al ~/.kaggle


total 16
drwxr-xr-x 2 root root 4096 Nov 14 21:30 .
drwx------ 1 root root 4096 Nov 14 21:30 ..
-rw------- 1 root root   70 Nov 14 21:32 kaggle.json


In [ ]:
# Dataset 1: nexuswho
!kaggle datasets download -d nexuswho/fabric-defects-dataset -p "{PROJECT_ROOT}/data/raw/fabric_defects_dataset"
!unzip -q "{PROJECT_ROOT}/data/raw/fabric_defects_dataset/fabric-defects-dataset.zip" -d "{PROJECT_ROOT}/data/raw/fabric_defects_dataset/"

# Dataset 2: yashrandive11
!kaggle datasets download -d yashrandive11/fabricfaultdetection -p "{PROJECT_ROOT}/data/raw/fabricfaultdetection"
!unzip -q "{PROJECT_ROOT}/data/raw/fabricfaultdetection/fabricfaultdetection.zip" -d "{PROJECT_ROOT}/data/raw/fabricfaultdetection/"


Dataset URL: https://www.kaggle.com/datasets/nexuswho/fabric-defects-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
100% 2.09G/2.09G [00:14<00:00, 176MB/s]
100% 2.09G/2.09G [00:14<00:00, 157MB/s]
Dataset URL: https://www.kaggle.com/datasets/yashrandive11/fabricfaultdetection
License(s): unknown
  0% 0.00/10.9M [00:00<?, ?B/s]
100% 10.9M/10.9M [00:00<00:00, 328MB/s]


Merge Both Datasets for Metadata

In [ ]:
import os
import glob
import pandas as pd

records = []

# Dataset 1
dataset1_path = f"{PROJECT_ROOT}/data/raw/fabric_defects_dataset/"
for img in glob.glob(dataset1_path + "**/*.jpg", recursive=True):
    label = os.path.basename(os.path.dirname(img))
    records.append([img, label, "fabric_defects_dataset"])

# Dataset 2
dataset2_path = f"{PROJECT_ROOT}/data/raw/fabricfaultdetection/"
for img in glob.glob(dataset2_path + "**/*.jpg", recursive=True):
    label = os.path.basename(os.path.dirname(img))
    records.append([img, label, "fabricfaultdetection"])

# Create DataFrame
df = pd.DataFrame(records, columns=["filepath", "label", "dataset_source"])
df.head()


,filepath,label,dataset_source
0,/content/drive/MyDrive/FabricDefect/data/raw/f...,Vertical,fabric_defects_dataset
1,/content/drive/MyDrive/FabricDefect/data/raw/f...,Vertical,fabric_defects_dataset
2,/content/drive/MyDrive/FabricDefect/data/raw/f...,Vertical,fabric_defects_dataset
3,/content/drive/MyDrive/FabricDefect/data/raw/f...,Vertical,fabric_defects_dataset
4,/content/drive/MyDrive/FabricDefect/data/raw/f...,Vertical,fabric_defects_dataset


Stratified Train/Val/Test Split

In [ ]:
import pandas as pd
import numpy as np

# Split 70% train, 15% val, 15% test
np.random.seed(42)

# First split: 70% train, 30% temp
train_indices = []
temp_indices = []

for label in df["label"].unique():
    label_indices = df[df["label"] == label].index.tolist()
    np.random.shuffle(label_indices)

    split_point = int(0.7 * len(label_indices))
    train_indices.extend(label_indices[:split_point])
    temp_indices.extend(label_indices[split_point:])

train_df = df.iloc[train_indices].copy()
temp_df = df.iloc[temp_indices].copy()

# Second split: 50% val, 50% test from temp (making 15% val, 15% test overall)
val_indices = []
test_indices = []

for label in temp_df["label"].unique():
    label_indices = temp_df[temp_df["label"] == label].index.tolist()
    np.random.shuffle(label_indices)

    split_point = int(0.5 * len(label_indices))
    val_indices.extend(label_indices[:split_point])
    test_indices.extend(label_indices[split_point:])

val_df = temp_df.iloc[val_indices].copy()
test_df = temp_df.iloc[test_indices].copy()

# Add split column
train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

# Combine into full dataframe
full_df = pd.concat([train_df, val_df, test_df]).reset_index(drop=True)

Save Metadata CSV

In [ ]:
csv_path = f"{PROJECT_ROOT}/data/metadata.csv"
full_df.to_csv(csv_path, index=False)
print(f"Metadata CSV saved at: {csv_path}")



Metadata CSV saved at: /content/drive/MyDrive/FabricDefect/data/metadata.csv


Copy Images into train/val/test Folders

In [ ]:
import shutil

def copy_files(df, split_name):
    split_dir = f"{PROJECT_ROOT}/data/{split_name}"
    for _, row in df[df['split']==split_name].iterrows():
        label_dir = os.path.join(split_dir, row['label'])
        os.makedirs(label_dir, exist_ok=True)
        shutil.copy(row['filepath'], label_dir)

for split in ["train", "val", "test"]:
    copy_files(full_df, split)


Verify

In [ ]:
import os

for split in ["train", "val", "test"]:
    path = f"{PROJECT_ROOT}/data/{split}"
    print(f"\n{split} folder classes and counts:")
    for cls in os.listdir(path):
        cls_path = os.path.join(path, cls)
        if os.path.isdir(cls_path):
            print(cls, len(os.listdir(cls_path)))



train folder classes and counts:
hole 197
fabricfaultdetection 55
stain 279
lines 110
defect free 1067
horizontal 95
Vertical 64

val folder classes and counts:
defect free 229
horizontal 21
stain 59
hole 42
lines 23
Vertical 14
fabricfaultdetection 12

test folder classes and counts:
defect free 229
stain 60
fabricfaultdetection 12
hole 42
lines 24
horizontal 20
Vertical 14


Create Dataset Loader

In [ ]:
%%writefile /content/drive/MyDrive/FabricDefect/utils/data_loader.py

import pandas as pd
from PIL import Image

class FabricDataset:
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        img = Image.open(row["filepath"]).convert("RGB")
        label = row["label"]
        return img, label, row["dataset_source"]


Writing /content/drive/MyDrive/FabricDefect/utils/data_loader.py


Test Loader

In [ ]:
import sys
sys.path.append(PROJECT_ROOT)

from utils.data_loader import FabricDataset

dataset = FabricDataset(f"{PROJECT_ROOT}/data/metadata.csv")
img, label, source = dataset[0]

img.show()
print("Label:", label)
print("Dataset source:", source)

Label: hole
Dataset source: fabric_defects_dataset
